# 02 – Data integration

**Goal:** Combine the four sources into consistent tables with common keys and document every join: key, rows before and after, and the reason for every lost record.
**Input:** `data/raw/`, `data/interim/jolpica_*.parquet`, `data/interim/wikipedia_circuits.parquet` · **Output:** `data/interim/{circuits,races,drivers,driver_map,results,laps,weather}.parquet`, `reports/logs/integration_log.csv`
**Assessment items:** P2 (preparation and integration of at least two sources, documented)

| Step | Left | Right | Key | Method |
|---|---|---|---|---|
| 1 | FastF1 sessions | Jolpica calendar | season + round | exact join, then name/date check |
| 2 | Jolpica circuits | GeoJSON locations | coordinates | nearest point (haversine) ≤ 5 km |
| 2b | Jolpica circuits | Wikipedia circuit list | circuit name (+ town) | fuzzy match (rapidfuzz) + manual correction table |
| 3 | FastF1 drivers | Jolpica drivers | season + round + car number | exact join, check against FastF1 DriverId and abbreviation |
| 4 | Jolpica results | Jolpica qualifying | season + round + driverId | left join |
| 5 | FastF1 laps | driver mapping (step 3) | season + round + car number | exact join; 2018 tyre names converted to race-relative SOFT/MEDIUM/HARD |
| 6 | FastF1 weather | laps | session time | as-of join (backward) when building pairs, notebook 04 |

In [1]:
import sys
from pathlib import Path

# Make the repository root importable, whether Jupyter starts in the root or in notebooks/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

In [2]:
from src.config import INTERIM_DIR, SEASONS
from src.integrate import integrate

jolpica = {name: pd.read_parquet(INTERIM_DIR / f"jolpica_{name}.parquet")
           for name in ("races", "results", "qualifying")}
wiki = pd.read_parquet(INTERIM_DIR / "wikipedia_circuits.parquet")

tables, log = integrate(SEASONS, jolpica, wiki)

## 1. Integration log

In [3]:
with pd.option_context("display.max_colwidth", None):
    display(log.style.hide(axis="index").set_properties(**{"text-align": "left"}))

step,rows_before,rows_after,change,note
1 races: FastF1 sessions <-> Jolpica calendar,196,192,-4,"key (season, round); only Jolpica (no FastF1 race data): 2026 R20 Brazilian Grand Prix, 2026 R21 Las Vegas Grand Prix, 2026 R22 Qatar Grand Prix, 2026 R23 Abu Dhabi Grand Prix"
1b races: consistency check name/date,192,192,0,0 races with name similarity < 80 or date difference > 1 day
2 circuits: Jolpica circuitId <-> GeoJSON location (nearest point),33,33,0,matched if distance <= 5.0 km; unmatched: []; max matched distance 1.03 km
2b circuits: Jolpica circuit name <-> Wikipedia list (fuzzy),33,33,0,"rapidfuzz token_set_ratio >= 85, or >= 50 with identical town, else manual table; {'fuzzy name': 31, 'fuzzy name + town': 2}"
2c circuits: geometry features from GeoJSON layout,33,33,0,projected to local UTM zone; straight = consecutive segments within 10 deg of the running direction; layout length vs. Wikipedia length: max. deviation 0.4 %
3 drivers: FastF1 car number per race <-> Jolpica driverId,3876,3876,0,"key (season, round, number); unmatched 0; driverId differs from FastF1 DriverId: 0; abbreviation differs from Jolpica code: 0"
"3b driver_map: unique (season, abbreviation)",198,197,-1,"abbreviations with several numbers/drivers in a season (first kept): [{'season': 2024, 'abbreviation': 'BEA', 'driver_number': 38, 'driver_id': 'bearman'}, {'season': 2024, 'abbreviation': 'BEA', 'driver_number': 50, 'driver_id': 'bearman'}]"
4 results: race result + qualifying position,3876,3876,0,"left join on (season, round, driverId); 9 drivers without qualifying position (quali_gap will be missing for them)"
5a laps: tyre compounds -> race-relative SOFT/MEDIUM/HARD,211297,211297,0,20 races with absolute 2018 names converted (softest used dry compound = SOFT); races without exactly 3 dry compounds: none; laps without known compound: 1288
"5 laps: FastF1 laps -> driver_id, seconds",211297,211297,0,"driver not mapped: 0; lap number missing: 0; duplicate (race, driver, lap): 0; info: position missing 319, IsAccurate=False 30299"


## 2. Circuit matching (GeoJSON and Wikipedia)

In [4]:
tables["circuit_matching"][["circuit_id", "circuit_name", "locality", "geojson_id",
                            "geo_distance_km", "wiki_name", "wiki_match", "wiki_score",
                            "circuit_type", "length_km", "layout_length_m", "turns",
                            "longest_straight_m"]]

,circuit_id,circuit_name,locality,geojson_id,geo_distance_km,wiki_name,wiki_match,wiki_score,circuit_type,length_km,layout_length_m,turns,longest_straight_m
0,albert_park,Albert Park Grand Prix Circuit,Melbourne,au-1953,0.45,Albert Park Circuit,fuzzy name,100.000000,street,5.278,5273.4,16,804.5
1,bahrain,Bahrain International Circuit,Sakhir,bh-2002,0.22,Bahrain International Circuit,fuzzy name,100.000000,race,5.412,5408.7,15,1120.9
2,shanghai,Shanghai International Circuit,Shanghai,cn-2004,0.15,Shanghai International Circuit,fuzzy name,100.000000,race,5.451,5444.6,16,1211.5
3,baku,Baku City Circuit,Baku,az-2016,1.03,Baku City Circuit,fuzzy name,100.000000,street,6.003,5989.9,20,1215.9
4,catalunya,Circuit de Barcelona-Catalunya,Barcelona,es-1991,0.21,Circuit de Barcelona-Catalunya,fuzzy name,100.000000,race,4.657,4665.4,14,1078.7
5,monaco,Circuit de Monaco,Monte Carlo,mc-1929,0.72,Circuit de Monaco,fuzzy name,100.000000,street,3.337,3326.5,19,274.3
6,villeneuve,Circuit Gilles Villeneuve,Montreal,ca-1978,0.69,Circuit Gilles Villeneuve,fuzzy name,100.000000,street,4.361,4364.5,14,685.5
7,ricard,Circuit Paul Ricard,Le Castellet,fr-1969,0.27,Circuit Paul Ricard,fuzzy name,100.000000,race,5.842,5827.6,15,1045.1
8,red_bull_ring,Red Bull Ring,Spielberg,at-1969,0.46,Red Bull Ring,fuzzy name,100.000000,race,4.318,4309.0,10,712.9
9,silverstone,Silverstone Circuit,Silverstone,gb-1948,0.73,Silverstone Circuit,fuzzy name,100.000000,race,5.891,5887.2,18,778.1


## 3. Data quality of the lap table

In [5]:
laps = tables["laps"]
quality = pd.Series({
    "laps": len(laps),
    "position missing": int(laps["position"].isna().sum()),
    "lap time missing": int(laps["lap_time_s"].isna().sum()),
    "IsAccurate = False": int((~laps["is_accurate"].astype(bool)).sum()),
    "pit-in laps": int(laps["pit_in"].sum()),
    "pit-out laps": int(laps["pit_out"].sum()),
})
quality.to_frame("count")

,count
laps,211297
position missing,319
lap time missing,930
IsAccurate = False,30299
pit-in laps,6992
pit-out laps,6880


In [6]:
# Race-relative compound per season (2018 converted from absolute names)
laps.assign(season=laps["race_id"] // 100).pivot_table(
    index="season", columns="compound", values="lap_number", aggfunc="count", fill_value=0)

compound,HARD,INTERMEDIATE,MEDIUM,SOFT,WET
season,,,,,
2018,9001,32,7442,5495,3
2019,7667,685,9159,6088,77
2020,6186,973,6899,4132,160
2021,10481,1730,8531,2922,32
2022,8035,1446,9204,4182,376
2023,11827,724,8311,3475,48
2024,13652,2177,8940,1798,37
2025,10850,1598,10525,3305,0
2026,8802,136,7381,5485,0


## 4. Resulting tables

In [7]:
pd.Series({name: len(df) for name, df in tables.items()
           if name != "circuit_matching"}).to_frame("rows")

,rows
circuits,33
races,192
drivers,45
driver_map,197
results,3876
laps,211297
weather,28923
